In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [2]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
from siuba import *

fs = get_fs()

In [3]:
import altair as alt
import folium

In [4]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [5]:
import _replica_utils

In [6]:
from IPython.display import HTML

In [7]:
from calitp_data_analysis import calitp_color_palette as cp

In [8]:
pd.set_option("display.max_columns", None)

In [9]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"


In [10]:
display(HTML("<h1>Origin-Destination Big Data Analysis: SCAG POIs</h1>"))

In [11]:
display(HTML("This analysis uses Replica Data to identify the top trip start points within a Regional Transportation Planning Agency (RTPA) asnd determine what types of trips are occuring when and where."))

In [12]:
shape_data_name = "origins/replica-stm_origin_la-03_06_26-origin_layer.zip"

origins_name = "replica-stm_origin_la-03_06_26-trips_dataset.zip"

In [13]:
origins = _replica_utils.read_in_and_prep_replica_data_w_shp(origins_name, shape_data_name, file_type="")

In [14]:
# len(origins)

In [15]:
display(HTML("<h2><strong>Trip Counts</strong></h2>"))

In [16]:
display(HTML(f"The number of trips Traveling <strong>From</strong> the top 20 locations is <strong>{len(origins)}</strong>"))


In [17]:
assert origins.origin_bgrp_2020.nunique() == 20

In [18]:
origin_tracts_list = origins.origin_bgrp_2020.unique().tolist()

In [19]:
summary = _replica_utils.return_score_summary_single_df(origins, origin_tracts_list, "origin_bgrp_2020")

In [20]:
summary.columns = [col.replace('_', ' ').title() for col in summary.columns]


In [21]:
summary_melt =  pd.melt(
        summary,
        id_vars=["Trip Grouping"],
        value_vars=['Trip Grouping', 'Total Trips',
                    'N Auto Trips', 'Pct Auto Trips',
                    'N Tranist Trips', 'Pct Transit Trips',
                    'N Walking Trips', 'Pct Walking Trips',
                    'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
                    'Auto Mean Miles','Auto Median Miles', 'Auto Max Miles',
                    'Transit Mean Minutes','Transit Median Minutes', 'Transit Max Minutes',
                    'Transit Mean Miles','Transit Median Miles', 'Transit Max Miles',
                    'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes',
                    'Walking Mean Miles','Walking Median Miles', 'Walking Max Miles',
                   ],
        var_name="Metric",  # New column for original measurement names
        # value_name="_Value"
)

In [22]:
display(HTML("<h2><strong>Trip Type Summaries</strong></h2>"))

In [23]:
list_of_dfs = []


for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    modes_breakdown_subset = _replica_utils.get_mode_split(origins_subset, "origin_bgrp_2020")

    list_of_dfs.append(modes_breakdown_subset)

modes_breakdown = pd.concat(list_of_dfs, ignore_index=True)

In [24]:
modes_breakdown.columns = [col.replace('_', ' ').title() for col in modes_breakdown.columns]

In [87]:
# modes_breakdown.sample()

In [26]:
alt.Chart(modes_breakdown).mark_bar().encode(
    x=alt.X("Trip Grouping:O", title = "Trip Origin"),
    y=alt.Y("Total Trips:Q", title="Total Number of Trips"),
    color=alt.Color("Mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    tooltip = ['Total Trips', 'Mode']
    ).properties(
    width=800,  
    height=300,
    title='Modes by Trip Type')

alt.Chart(...)

In [27]:
# alt.Chart(modes_breakdown).mark_bar().encode(
#     x=alt.X('Trip Grouping:O', title ="Trip Origin"),
#     y= alt.Y('Pct Trips:Q', title="Pct of Total Trips"),
#     color=alt.Color('Trip Grouping:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Grouping')),
#     column= alt.Column('Mode:N', title="Mode"),
#     tooltip=['Pct Trips', 'Mode', 'Trip Grouping']
# ).properties(width = 100, height = 400, title = "Modes by Trip Type")

In [28]:
display(HTML("<h3><strong>Closer Look at Auto, Transit & Walking Trips</strong></h3>"))

In [29]:
display(HTML("<strong>Tip:</strong> Use the dropdown menu to select different metrics to see on the chart."))

In [30]:
metrics_list = summary_melt["Metric"].unique().tolist()

metrics_dropdown = alt.binding_select(
        options=metrics_list,
        name="Metrics: ",
    )
    # Column that controls the bar charts
xcol_param = alt.selection_point(
        fields=["Metric"], value=metrics_list[0], bind=metrics_dropdown
    )

chart = (alt.Chart(summary_melt, title="Metric by Trip Types")
        .mark_bar()
        .encode(
            x=alt.X("value"),
            y=alt.Y("Trip Grouping"),
            color=alt.Color("Trip Grouping",
                                  scale=alt.Scale(
                                      range=cp.CALITP_CATEGORY_BRIGHT_COLORS))
        ).properties(width=400, height=350)
    ).add_params(xcol_param).transform_filter(xcol_param)

display(HTML("""
<style>
form.vega-bindings {
  position: absolute;
  left: 0px;
  top: -0px;
}
</style>
"""))

(chart)

alt.Chart(...)

In [31]:
display(HTML("<br>"
            "<br>"))

In [32]:
summary_long_all_miles = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Miles', 'Auto Median Miles', 'Auto Max Miles', 
        'Transit Mean Miles', 'Transit Median Miles', 'Transit Max Miles',
        'Walking Mean Miles', 'Walking Median Miles', 'Walking Max Miles'],
        var_name="Metric",  # New column for original measurement names
        value_name="Miles")

In [33]:
summary_long_all_min = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
        'Transit Mean Minutes', 'Transit Median Minutes', 'Transit Max Minutes',
        'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes'],
        var_name="Metric",  # New column for original measurement names
        value_name="Mintues")

In [88]:
display(HTML("<h3><strong>Trip Length</strong></h3>"))

In [35]:
alt.Chart(summary_long_all_min).mark_bar().encode(
    x="Mintues:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="Trip Grouping:O",
    facet=alt.Facet('Trip Grouping:O').columns(4),
).properties(title="Travel Length by Trip Type & Mode")

alt.Chart(...)

In [36]:
alt.Chart(summary_long_all_miles).mark_bar().encode(
    x="Miles:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="Trip Grouping:O",
    facet=alt.Facet('Trip Grouping:O').columns(4),
).properties(title="Travel Distance by Trip Type & Mode")

alt.Chart(...)

In [89]:
display(HTML("<h3><strong>Trip Timing</strong></h3>"))

In [38]:
display(HTML("<strong>Tip:</strong> You can zoom into each graph to better see the timing of the trips by mode"))

In [39]:
import datetime

In [40]:
def add_hour_summary_by_mode(df, grouping_col, grouping_list, mode_col): 
    
    time_start = []
    time_end = []
    
    for grouping_value in grouping_list:

        df_subset_start = df[df[grouping_col]== grouping_value]
        df_subset_end = df[df[grouping_col]== grouping_value]

        df_subset_start['trip_start_time'] = pd.to_datetime(df_subset_start['trip_start_time'])
        start_time_bins = df_subset_start.set_index('trip_start_time')
        result_start_time_bins = start_time_bins.groupby([mode_col, grouping_col]).resample('30T').size().reset_index(name='trip_count')

        df_subset_end['trip_start_time'] = pd.to_datetime(df_subset_end['trip_start_time'])
        end_time_bins = df_subset_end.set_index('trip_start_time')
        result_end_time_bins = end_time_bins.groupby([mode_col, grouping_col]).resample('30T').size().reset_index(name='trip_count')

        time_start.append(result_start_time_bins)
        time_end.append(result_end_time_bins)
            

    start_time_all = pd.concat(time_start)
    end_time_all = pd.concat(time_end)

    return start_time_all, end_time_all
       

In [41]:
start, end = add_hour_summary_by_mode(origins, "origin_bgrp_2020", origin_tracts_list, "primary_mode")

In [85]:
alt.Chart(start).mark_bar().encode(
    x=alt.X("trip_start_time:T",axis=alt.Axis(format='%H:%M')),
    y=alt.Y("trip_count:Q", title="Number or Trips"),
    color=alt.Color("primary_mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="origin_bgrp_2020:O",
    facet=alt.Facet('origin_bgrp_2020:O').columns(4)).properties(title="Trip Start Time by Primary Mode")

alt.Chart(...)

In [86]:
alt.Chart(end).mark_bar().encode(
    x=alt.X("trip_start_time:T",axis=alt.Axis(format='%H:%M')),
    y=alt.Y("trip_count:Q", title= "Number of Trips"),
    color=alt.Color("primary_mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="origin_bgrp_2020:O",
    facet=alt.Facet('origin_bgrp_2020:O').columns(4)).properties(title="Trip End Time by Primary Mode")

alt.Chart(...)

In [44]:
melt_dfs = []
time_duration_dfs =  []

for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    origins_subset_time_melt, origins_subset_time_melt_duration = _replica_utils.return_time_metrics(origins_subset, "trip_start_time", "trip_end_time")
    
    melt_dfs.append(origins_subset_time_melt)
    time_duration_dfs.append(origins_subset_time_melt_duration)



time_melt = pd.concat(melt_dfs, ignore_index=True)
time_duration_melt = pd.concat(time_duration_dfs, ignore_index=True)


In [45]:
# alt.Chart(time_melt).mark_circle(size=150).encode(
#     x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Time'],
# ).properties(height=400, width=600, title="Trips Start and End Times")

In [46]:
# alt.Chart(time_duration_melt).mark_circle(size=150).encode(
#     x=alt.X('Value:Q', title="Minutes"),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Value']
# ).properties(height=400, width=600, title="Trip Duration for Trips")

In [47]:
n_trips_origin = (origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry"]).agg(
        {"activity_id": "nunique"}).reset_index()
n_trips_origin = n_trips_origin.set_geometry("geometry")


In [48]:
n_trips_dest = (origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry"]).agg(
        {"activity_id": "nunique"}).reset_index()
n_trips_dest = n_trips_dest.set_geometry("geometry")


In [90]:
display(HTML("<h3><strong>Trips Duration</strong></h3>"))

In [53]:
# origin_tracts_list

In [54]:
agg_stats_results = []

for tract in origin_tracts_list:
    
    origins_subset = origins[origins["origin_bgrp_2020"] == tract ]
    origins_subset = origins_subset[origins_subset["primary_mode"]!="other_travel_mode"]
    
    k = 1.5
    group_by_column = "primary_mode"
    value_column = "trip_duration_minutes"

    agg_stats = origins_subset.groupby(group_by_column)[value_column].describe()
    
    agg_stats["iqr"] = agg_stats["75%"] - agg_stats["25%"]
    agg_stats["min_"] = agg_stats["25%"] - k * agg_stats["iqr"]
    agg_stats["max_"] = agg_stats["75%"] + k * agg_stats["iqr"]
    data_points = origins_subset[[value_column, group_by_column]].merge(agg_stats.reset_index()[[group_by_column, "min_", "max_"]])
    
    # This will be the lower end of the whisker
    agg_stats["lower"] = (
        data_points[data_points[value_column] >= data_points["min_"]]
        .groupby(group_by_column)[value_column]
        .min()
    )
    # This will be the upper end of the whisker
    agg_stats["upper"] = (
        data_points[data_points[value_column] <= data_points["max_"]]
        .groupby(group_by_column)[value_column]
        .max()
    )
    
    agg_stats = agg_stats.reset_index()

    agg_stats['origin'] = tract

    agg_stats_results.append(agg_stats)

agg_stats_all = pd.concat(agg_stats_results, ignore_index=True)

In [55]:
# agg_stats_all.sample()

In [56]:
import ipywidgets as widgets
from ipywidgets import interact


In [57]:
dropdown = widgets.Dropdown(
    options=origin_tracts_list,
    value=origin_tracts_list[0],
    description='Origin:',
    disabled=False,
)
output = widgets.Output()


In [58]:
def make_chart(tract):

    df = agg_stats_all[agg_stats_all['origin']==tract]
    
    base = alt.Chart(df).encode(y = alt.Y("primary_mode:N", title="Primary Mode")).properties(title= f"Trip Durations for {tract}")

    rules = base.mark_rule().encode(
        x= alt.X("lower").title("Trip Duration (Minutes)"),
        x2="upper",)
    
    bars = base.mark_bar(size=14).encode(
        x="25%",
        x2="75%",
        color=alt.Color("primary_mode").legend(None),)
    
    ticks = base.mark_tick(color="white", size=14).encode(x="50%")
    
    outliers = base.transform_flatten(flatten=["outliers"]
                                     ).mark_point(style="boxplot-outliers").encode(
        x="outliers:Q",
        color="primary_mode").add_params(xcol_param).transform_filter(xcol_param)

    chart = (rules + bars + ticks).interactive()
   
    return chart

In [59]:
interact(make_chart, tract=origin_tracts_list);


interactive(children=(Dropdown(description='tract', options=('1 (Tract 127, San Bernardino, CA)', '1 (Tract 92…

In [60]:
### non drop down version
# for tract in origin_tracts_list:

#     agg_stats_all_subset = agg_stats_all[agg_stats_all["origin"] == tract ]


#     base = alt.Chart(agg_stats_all_subset).encode(y = alt.Y("primary_mode:N", title="Primary Mode")).properties(title= f"Trip Durations for {tract}")

#     rules = base.mark_rule().encode(
#         x= alt.X("lower").title("Trip Duration (Minutes)"),
#         x2="upper",)
    
#     bars = base.mark_bar(size=14).encode(
#         x="25%",
#         x2="75%",
#         color=alt.Color("primary_mode").legend(None),)
    
#     ticks = base.mark_tick(color="white", size=14).encode(
#         x="50%"
#     )
    
#     outliers = base.transform_flatten(
#         flatten=["outliers"]
#     ).mark_point(
#         style="boxplot-outliers"
#     ).encode(
#         x="outliers:Q",
#         color="primary_mode",
#     ).add_params(xcol_param).transform_filter(xcol_param)

#     chart = (rules + bars + ticks)

#     display(chart)

In [84]:
display(HTML("<h2><strong>Trips by Census Block Groups</strong></h2>"))

In [ ]:
n_trips_origin = n_trips_origin.rename(columns={"activity_id":"Number of Trips", "origin_bgrp_2020":"Origin Census BlockGroup"})

n_trips_dest = n_trips_dest.rename(columns={"destination_bgrp_2020":"Destination Census BlockGroup","activity_id":"Number of Trips"})

In [82]:
display(HTML("<h4>Trips by Origin</h4>"))

In [ ]:
m = n_trips_origin.explore(name="Trips from Origins", column="Number of Trips", 
                cmap="YlOrRd")
display(m)

In [77]:
# display(HTML("<h4>Trips by Destination</h3>"))

In [78]:
# n_trips_dest.sample()

In [79]:
# m = n_trips_dest.explore(name="Trips by Destination", column="Number of Trips", cmap="YlOrRd")
# m

In [80]:
# n_trips_dest_mode = (
#     origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
#         n_trips=("activity_id", "nunique"),
        
#         trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#         trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
        
#         trip_distance_miles_median=('trip_distance_miles', 'median'),
#         trip_distance_miles_mean=('trip_distance_miles', 'mean'),
        
#     #     trip_start_time_mean=('trip_start_time', 'mean'),
#     #     trip_start_time_median=('trip_start_time', 'median'),
        
#     #     trip_end_time_mean=('trip_end_time', 'mean'),
#     #     trip_end_time_median=('trip_end_time', 'median'),
    
#             ).reset_index()
    
# n_trips_dest_mode = n_trips_dest_mode.set_geometry("geometry")

In [81]:
# n_trips_origin_mode = (
#     origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry", "primary_mode"]).agg(
#         n_trips=("activity_id", "nunique"),
        
#         trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#         trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
        
#         trip_distance_miles_median=('trip_distance_miles', 'median'),
#         trip_distance_miles_mean=('trip_distance_miles', 'mean'),
        
#     #     trip_start_time_mean=('trip_start_time', 'mean'),
#     #     trip_start_time_median=('trip_start_time', 'median'),
        
#     #     trip_end_time_mean=('trip_end_time', 'mean'),
#     #     trip_end_time_median=('trip_end_time', 'median'),
    
#             ).reset_index()
        
# n_trips_origin_mode = n_trips_origin_mode.set_geometry("geometry")

In [66]:
# alt.Chart((n_trips_dest_mode>>filter(_.destination_bgrp_2020 == "1 (Tract 3570, Contra Costa, CA)"))).mark_bar().encode(
#     x=alt.X('primary_mode', title ="Trip Mode"),
#     y= alt.Y('n_trips', title="Number of Trips"),
#     # color=alt.Color('destination_bgrp_2020:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Type')),
# ).properties(width = 500, height = 400, title = "Modes by Trip Type")

In [67]:
# n_trips_from_cp_mode.sample()

In [68]:
# unique_modes = origins.primary_mode.unique()

In [69]:
# unique_modes

In [70]:
# unique_modes = ['private_auto', 'public_transit', 'walking']

In [71]:
# display(HTML("<h2>Trips Originating in the Top 20 tracts in the SCAG Area</h2>"))

In [72]:
# ##hashing out for now for saving
# replica_utils.return_mode_map(n_trips_to_cp_mode, routes, unique_modes, "to")

In [73]:
# display(HTML("<h4>Trips Destinations of the Top 20 tracts in the SCAG area</h4>"))

In [74]:
###hashing out for now for saving
# replica_utils.return_mode_map(n_trips_from_cp_mode, routes, unique_modes, "from")

In [75]:
display(HTML("<h2>Raw Trip Data Sample</h2>"))

In [76]:
origins.sample(3)

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,tour_type,trip_start_local_hour,trip_end_local_hour,origin_zcta_2020,destination_zcta_2020,geometry
160261,16821352703994125634,"1 (Tract 9800.13, Los Angeles, CA)","9800.13 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 1153.02, Los Angeles, CA)","1153.02 (Los Angeles, CA)","Los Angeles County, CA",California,public_transit,home,work,16:46:00,18:13:51,87,35.7,unknown_vehicle_type,unknown_fuel_type,bus,Metro - Los Angeles,Metro G Line 901,retail,retail,multi_family,multi_family,9247431580277607057,17351022491180721854,48.0,female,hispanic_or_latino_origin,employed,in_person,11594.0,public_transit,5,27294.0,two,core,naics56,multiple_units,not_attending_school,k_12,renter,spanish,"1 (Tract 1153.02, Los Angeles, CA)","1153.02 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 9800.13, Los Angeles, CA)","9800.13 (Los Angeles, CA)","Los Angeles County, CA",California,commute,16,18,90245,91324,"POLYGON ((-118.39628 33.92702, -118.39627 33.9..."
73357,11524220351206227442,"1 (Tract 250, San Bernardino, CA)","250 (San Bernardino, CA)","San Bernardino County, CA",California,"1 (Tract 250, San Bernardino, CA)","250 (San Bernardino, CA)","San Bernardino County, CA",California,other_travel_mode,work,home,07:37:00,08:33:47,56,21.2,NaN,unknown_fuel_type,NaN,NaN,NaN,single_family,single_family,civic_institutional,civic_institutional,14266906277857128299,13187314634385926804,38.0,male,white_not_hispanic_or_latino,employed,in_person,132407.0,biking,4,154113.0,two,core,not_working,single_family,not_attending_school,advanced_degree,renter,english,"1 (Tract 250, San Bernardino, CA)","250 (San Bernardino, CA)","San Bernardino County, CA",California,"1 (Tract 250, San Bernardino, CA)","250 (San Bernardino, CA)","San Bernardino County, CA",California,commute,7,8,92310,Out of Region,"POLYGON ((-116.92734 35.33608, -116.92734 35.3..."
49451,2436671398288069585,"1 (Tract 2074, Los Angeles, CA)","2074 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 1396, Los Angeles, CA)","1396 (Los Angeles, CA)","Los Angeles County, CA",California,private_auto,shop,work,14:51:00,15:37:32,46,22.9,unknown_vehicle_type,other_non_bev,NaN,NaN,NaN,retail,retail,retail,retail,1416053535053746708,9601900182326321337,32.0,male,white_not_hispanic_or_latino,employed,in_person,48308.0,private_auto,3,48308.0,two,core,naics2389,multiple_units,not_attending_school,high_school,renter,indo_european,"1 (Tract 1278.06, Los Angeles, CA)","1278.06 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 2074, Los Angeles, CA)","2074 (Los Angeles, CA)","Los Angeles County, CA",California,commute,14,15,90012,91316,"POLYGON ((-118.25333 34.05864, -118.25307 34.0..."
